In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using localhost Nocodb")
else:
    print("using AWS Nocodb")

using AWS Nocodb


In [9]:
from birddog.database import Database
from birddog.wiki import page_kind

2026-04-08 13:31:27,692 [INFO] Translation is enabled. Using GCP translator
2026-04-08 13:31:27,693 [INFO] Using Google Cloud translation API
2026-04-08 13:31:27,693 [INFO] GoogleCloudTranslator using REST API


In [15]:
page_kind("ДАКО/Д")

'archive'

In [4]:
db = Database()

2026-04-08 13:23:35,810 [INFO] 
AdaptiveThrottle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     4.66    39.00       0.00           24


In [17]:
recs, _ = db.scan("Pages", view_name="Empty Level", fields=["url","title"], limit=500)

In [18]:
len(recs)

500

In [19]:
recs[0]

{'Id': 7225,
 'title': 'ЦДІАК/1166/1/26',
 'url': 'https://uk.wikisource.org/wiki/Архів:ЦДІАК/1166/1/26'}

In [20]:
page_kind(recs[0]["title"])

'case'

In [32]:
cursor = None
result = []
while True:
    recs, cursor = db.scan("Pages", view_name="Empty Level", fields=["url","title"], limit=500, cursor=cursor)
    if not recs:
        break
    print(recs[0], cursor)
    result.extend([{**rec, "level": page_kind(rec["title"])} for rec in recs])
    print(len(result))

2026-04-08 13:50:38,877 [INFO] 
AdaptiveThrottle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   40.00     0.51    39.00       0.00           24
{'Id': 11652, 'title': 'ЦДІАК/127/1016/385', 'url': 'https://uk.wikisource.org/wiki/Архів:ЦДІАК/127/1016/385'} 500
500
{'Id': 12640, 'title': 'ЦДІАК/11/1/4', 'url': 'https://uk.wikisource.org/wiki/Архів:ЦДІАК/11/1/4'} 1000
1000
{'Id': 13422, 'title': 'ДАЧкО/18/1/454', 'url': 'https://uk.wikisource.org/wiki/Архів:ДАЧкО/18/1/454'} 1500
1500
{'Id': 13934, 'title': 'ДАЧкО/8/2/177', 'url': 'https://uk.wikisource.org/wiki/Архів:ДАЧкО/8/2/177'} 2000
2000
{'Id': 14408, 'title': 'ДАЧкО/832/1/421', 'url': 'https://uk.wikisource.org/wiki/Архів:ДАЧкО/832/1/421'} 2500
2500
{'Id': 14924, 'title': 'ДАЧкО/832/1/560', 'url': 'https://uk.wikisource.org/wiki/Архів:ДАЧкО/832/1/560'} 300

KeyboardInterrupt: 

In [34]:
by_id = { r["Id"]: r for r in result }

In [35]:
len(by_id)

27902

In [ ]:
db.write("Pages", list(by_id.values()))

2026-04-08 13:54:21,978 [INFO] 
AdaptiveThrottle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   42.00     0.34    39.00       0.00           24
2026-04-08 13:55:22,159 [INFO] 
AdaptiveThrottle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   47.00     4.67    39.00       0.00           24


In [31]:
len({r["Id"] for r in result})

27902